# Semaine 2 — Jour 7 : Agent autonome

Ce notebook accompagne le Markdown du jour.

Objectif : manipuler un agent autonome mono-agent contrôlé, sans API externe.

## 1. Boucle agentique

La boucle à maîtriser est :

```text
objective -> plan -> act -> observe -> update state -> decide
```

Un agent autonome doit aussi savoir s'arrêter.

In [ ]:
from pathlib import Path
import sys

lab_path = Path.cwd() / "book" / "week02" / "day07" / "labs"
if lab_path.exists():
    sys.path.insert(0, str(lab_path))

from autonomous_agent import AutonomousSupportAgent, build_default_registry, RunStatus, TaskStatus

In [ ]:
registry = build_default_registry()
agent = AutonomousSupportAgent(registry)

state = agent.start("user-1", "Je veux un remboursement pour ORDER-1234")
state = agent.run(state, approve_sensitive_actions=False)

state.status, state.missing_inputs

## 2. Inspection du plan

In [ ]:
[(task.id, task.name, task.tool_name, task.status.value, task.requires_approval) for task in state.tasks]

## 3. Exécution avec approbation

L'action sensible peut être exécutée uniquement si l'orchestrateur reçoit une approbation explicite.

In [ ]:
state2 = agent.start("user-1", "Je veux un remboursement pour ORDER-1234")
state2 = agent.run(state2, approve_sensitive_actions=True)

state2.status, state2.final_answer

## 4. Traces

In [ ]:
[(event.step, event.event_type, event.message) for event in state2.traces]

## 5. Exercice

Modifiez le lab pour empêcher l'exécution d'un outil lorsque `budget_used + cost > max_budget`.

# Section formateur — Corrigés et review

Cette section ne doit pas être incluse dans le notebook étudiant.

## Correction attendue — Garde-fou budget

La règle plus stricte consiste à anticiper le coût de l'action avant l'exécution.

```python
expected_cost = estimate_tool_cost(task.tool_name)
if state.budget_used + expected_cost > state.max_budget:
    stop()
```

Dans le lab simplifié, le coût est connu seulement après exécution. Une amélioration consiste à ajouter
`estimated_cost` sur `ToolDefinition`.

In [ ]:
agent = AutonomousSupportAgent(build_default_registry())
state = agent.start("user-1", "Je veux un remboursement pour ORDER-1234", max_budget=1)
final_state = agent.run(state, approve_sensitive_actions=True)

assert final_state.status == RunStatus.STOPPED
assert any(event.event_type == "budget_exceeded" for event in final_state.traces)
print("Budget guardrail OK")

## Points de review

- Vérifier que les apprenants distinguent state, memory et historique.
- Insister sur le fait qu'un prompt ne suffit pas à protéger une action sensible.
- Demander où placer l'appel LLM réel dans l'architecture.
- Demander comment auditer un incident à partir des traces.